# Recuperação de sessões pós-quânticas em IoT

Este notebook recalcula os números do artigo a partir das observações originais e permite repetir os experimentos com novas medições. As duas etapas são identificadas separadamente. A implementação está nas classes do simulador.

O `Tutorial_Execucao.ipynb` continua sendo o exemplo do simulador geral e não foi alterado.

## 1. Ambiente de execução

Para **consultar as tabelas de referência**, basta o ambiente Python do simulador. Para **repetir os experimentos TLS e Mininet-WiFi**, construa as imagens na raiz do repositório:

```bash
docker compose build notebook
docker build -t pqc-iot-tls:organizacao -f docker/tls/Dockerfile .
docker build -t pqc-iot-continuity:organizacao -f docker/continuity/Dockerfile .
docker compose -f docker-compose.yml -f docker/context_costs/compose.yml build notebook
docker compose -f docker-compose.yml -f docker/context_costs/compose.yml run --rm --service-ports notebook
```

Abra este notebook no kernel **Python 3 - PQC IoT Simulator**. Execute as células na ordem apresentada. O contêiner da pesquisa inclui OpenSSL 3.5.9 e os módulos TLS nativos. A execução de rede exige Linux, privilégios no contêiner e suporte a `mac80211_hwsim` no host.

In [ ]:
from pathlib import Path
from datetime import datetime, timezone
from IPython.display import display, Image
from pqc_iot_simulator import Simulation

simulacao = Simulation()
referencia = simulacao.recovery_reference()
display(referencia.verify())
identificador_reanalise = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
reanalise = simulacao.reanalyze_recovery(Path("results/reanalise_artigo") / identificador_reanalise)
display(reanalise.run())


## 2. Configurações comparadas

Todos os perfis usam TLS 1.3, autenticação mútua, AES-256-GCM/SHA-384 e 0-RTT desabilitado.

| Perfil | Estabelecimento de chaves | Autenticação |
|---|---|---|
| `classical` | X25519 | ECDSA P-256 |
| `hybrid` | X25519MLKEM768 | ECDSA P-256 |
| `pqc` | X25519MLKEM768 | ML-DSA-44 |

Os tratamentos são `full` (autenticação completa direta), `resume_volatile` (chave dos tickets perdida no reinício) e `resume_preserved` (chave preservada). As duas políticas de retomada usam a mesma construção de tickets. O banco de leituras é preservado nessa comparação.

## 3. Resultados originais: transmissão da raiz

A célula anterior recalcula as tabelas usando as observações por execução distribuídas com o repositório e verifica igualdade exata com as tabelas do artigo. Não executa novas conexões nem substitui a auditoria dos PCAP.

O experimento compara transmissão e omissão da CA raiz nos dois participantes, mantendo a raiz confiável. São 540 execuções em dez blocos, com falhas em 0%, 25% ou 100% das leituras elegíveis. Cada execução contém nove leituras com payload de 31 bytes. As tabelas abaixo mostram os resultados originais verificados.

In [ ]:
cadeias = referencia.table("chains")
display(cadeias.loc[
    (cadeias["profile"] == "pqc") & (cadeias["restart_fraction"] == 1),
    ["include_root", "treatment", "blocks", "median_ipv4_bytes_per_confirmed_reading"],
])
display(referencia.table("chain_comparisons").query("profile == 'pqc' and restart_fraction == 1"))

## 4. Resultados de referência: leituras por conexão

O experimento mantém 100 leituras de 512 bytes e limita cada conexão a 1, 10 ou 100 leituras. Compara ausência de falhas com reinício antes da 51ª leitura e supressão de sua primeira confirmação. A raiz não é transmitida, mas permanece confiável. São 540 execuções em dez blocos.

In [ ]:
amortizacao = referencia.table("amortization")
display(amortizacao.query("profile == 'pqc'")[[
    "recovery", "batch_size", "treatment", "blocks",
    "median_ipv4_bytes_per_confirmed_reading", "median_ms_per_reading",
]])

## 5. Repetir os experimentos com novas medições

`REPRODUZIR = True` executa o desenho completo no contêiner de pesquisa. Com dez blocos, a execução pode levar várias horas. Para recalcular somente os resultados originais e consultar as referências, use `False`.

Os novos dados são gravados em um diretório exclusivo, incluindo JSON, configurações, agendas, PCAP e verificações. Os arquivos originais não são substituídos. Ao final, o notebook compara as mesmas condições com o artigo e mostra as diferenças, inclusive quando muda o arredondamento publicado. Bytes IPv4 e tempos são medições e podem variar entre execuções.

`BLOCOS = 10` corresponde à pesquisa. Um valor menor representa outra amostra; a comparação completa com o artigo exige dez blocos. Use pelo menos dois para calcular os intervalos da continuidade.

In [ ]:
REPRODUZIR = True
BLOCOS = 10
identificador = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
estudo = simulacao.session_recovery(Path("results/reproducao") / identificador, blocks=BLOCOS)

if REPRODUZIR:
    display(estudo.preflight())
else:
    print("Consulta aos resultados de referência. Nenhum experimento novo foi executado.")

### 5.1 Ambiente e referências

Conferimos a disponibilidade do ambiente TLS e a integridade dos dados de referência antes da execução. Os cenários de identidade, persistência e interrupção são executados pelas classes de controle na etapa 5.6.


In [ ]:
if REPRODUZIR:
    display(estudo.preflight())
    assert referencia.verify()["status"] == "PASS"

### 5.2 Protótipo: chaves, contadores e interrupções

Executa as seis políticas e quatro falhas com dez repetições (240 execuções), seguidas dos onze cenários de interrupção e três modos com chaves novas (330 casos). O protótipo usa primitivas criptográficas reais e transporte lógico; seus bytes são distintos dos bytes TLS.

In [ ]:
if REPRODUZIR:
    display(estudo.run_prototype())

### 5.3 TLS: preservação do contexto e frequência de falhas

Executa 1.620 casos da matriz principal e 180 casos adicionais sob perdas de 1% e 5%, quando `BLOCOS = 10`. Conserva as agendas entre os comparadores, a ordem aleatorizada, os controles de duplicação e as tentativas até confirmação. Use pelo menos dois blocos para calcular os intervalos da análise de continuidade; um bloco serve apenas para verificação funcional das etapas individuais.

In [ ]:
if REPRODUZIR:
    display(estudo.run_continuity())

### 5.4 TLS: cadeia com e sem raiz

Repete a comparação dos três tratamentos e perfis, modificando apenas a transmissão da raiz. As cadeias efetivamente enviadas pelo cliente e pelo servidor são verificadas.

In [ ]:
if REPRODUZIR:
    display(estudo.run_chains())

### 5.5 TLS: amortização do handshake

Repete as cargas de 100 leituras com conexões de 1, 10 ou 100 leituras, com e sem recuperação. A captura inclui as tentativas e os encerramentos observados no servidor.

In [ ]:
if REPRODUZIR:
    display(estudo.run_amortization())

### 5.6 Tickets, identidade, persistência e checkpoint

Registra comprimentos públicos de certificados, sessões e tickets, incluindo um controle que altera somente a credencial do cliente. Verifica o vínculo entre o dispositivo e a identidade autenticada e interrompe a gravação antes/depois do commit e antes da confirmação. Diferencia encerramento com limpeza e término abrupto do processo; esses testes não representam queda física de energia.

In [ ]:
if REPRODUZIR:
    display(estudo.run_security())

## 6. Analisar e verificar os novos resultados

A análise conserva o bloco como unidade de repetição e calcula as diferenças dentro da mesma repetição. Confronta resultados, cadeias e capturas. A comparação final distingue reprodução das conclusões e igualdade dos números na precisão apresentada no artigo, sem ajustar os dados para forçar concordância.

In [ ]:
if REPRODUZIR:
    display(estudo.analyze_continuity())
    display(estudo.analyze_context())
    display(referencia.compare(estudo.directory / "context_analysis"))
    comparacao_artigo = estudo.compare_article()
    display(comparacao_artigo)


## 7. Gerar e consultar as figuras

As figuras usam as tabelas analisadas e o diagnóstico de tickets. Os gráficos permanecem coloridos. No modo de consulta, são mostradas as figuras de referência; ao reproduzir, são geradas figuras com os dados novos.

### Gráficos: cadeias, tickets e leituras por conexão

As seleções de dados, escalas e comandos de desenho estão nesta célula. As figuras são calculadas a partir das tabelas e verificações, sem valores de resultado inseridos manualmente.

In [ ]:
"""Exporta seis painéis dos testes de contexto, sem executar novos experimentos."""


import csv
import hashlib
import json
from pathlib import Path

import matplotlib

matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
from matplotlib.patches import Patch

from pqc_iot_simulator.pqc_iot_sim.utils.resources import DATA

ROOT = Path.cwd()
PROFILES = ("classical", "hybrid", "pqc")
NAMES = (
    "X25519\nECDSA P-256",
    "X25519MLKEM768\nECDSA P-256",
    "X25519MLKEM768\nML-DSA-44",
)
POLICIES = ("full", "resume_volatile", "resume_preserved")
LABELS = ("Completo", "Perdido", "Preservado")
COLORS = ("#315a91", "#b56a29", "#227a65")
FILES = (
    "cadeias_custos_absolutos.csv",
    "cadeias_comparacoes_mesma_repeticao.csv",
    "amortizacao_custos_absolutos.csv",
    "amortizacao_comparacoes_mesma_repeticao.csv",
)


def sha(path):
    return hashlib.sha256(path.read_bytes()).hexdigest()


def read(path):
    with path.open(newline="") as f:
        return list(csv.DictReader(f))


def one(rows, **where):
    def equal(actual, expected):
        if isinstance(expected, (int, float)) and not isinstance(expected, bool):
            return float(actual) == expected
        return actual == str(expected)

    found = [r for r in rows if all(equal(r[k], v) for k, v in where.items())]
    assert len(found) == 1, (where, len(found))
    assert int(found[0]["blocks"]) > 0
    return found[0]


def style(ax):
    ax.grid(axis="y", alpha=0.18, linewidth=0.6)
    ax.set_axisbelow(True)
    ax.spines[["top", "right"]].set_visible(False)


def key_title(ax):
    ax.set_title("X25519MLKEM768 + ML-DSA-44", fontsize=8, pad=6)


def panel_a(ax, chain, comparisons, amort, amort_cmp, diagnostic):
    x = np.arange(3)
    values = {}
    for j, (policy, color, label) in enumerate(zip(POLICIES, COLORS, LABELS)):
        ys = [
            float(
                one(
                    chain,
                    restart_fraction=1,
                    ack_fraction=1,
                    include_root=0,
                    profile=p,
                    treatment=policy,
                )["median_ipv4_bytes_per_confirmed_reading"]
            )
            / 1000
            for p in PROFILES
        ]
        values[policy] = ys
        ax.bar(x + (j - 1) * 0.25, ys, 0.23, color=color, label=label)
    ax.set_xticks(x, NAMES)
    ax.set_ylabel("IPv4 por leitura (kB)")
    ax.margins(y=0.2)
    ax.set_ylim(bottom=0)
    ax.set_title("Raiz omitida; ambas as falhas em 100%", fontsize=8, pad=6)
    ax.legend(ncol=3, fontsize=7.2, loc="upper left", columnspacing=0.7, handlelength=1)


def panel_b(ax, chain, comparisons, amort, amort_cmp, diagnostic):
    x = np.arange(3)
    series = (
        (1, "resume_volatile", COLORS[1], "//"),
        (0, "resume_volatile", COLORS[1], ""),
        (1, "resume_preserved", COLORS[2], "//"),
        (0, "resume_preserved", COLORS[2], ""),
    )
    for j, (root, policy, color, hatch) in enumerate(series):
        rows = [
            one(
                comparisons,
                restart_fraction=1,
                ack_fraction=1,
                include_root=root,
                profile=p,
                treatment=policy,
            )
            for p in PROFILES
        ]
        ys = [float(r["median_savings_pct"]) for r in rows]
        errors = [
            [y - float(r["savings_ci_low"]) for y, r in zip(ys, rows)],
            [float(r["savings_ci_high"]) - y for y, r in zip(ys, rows)],
        ]
        ax.bar(
            x + (j - 1.5) * 0.19,
            ys,
            0.18,
            color=color,
            hatch=hatch,
            edgecolor="white",
            linewidth=0.5,
            yerr=errors,
            capsize=1.5,
            error_kw={"elinewidth": 0.6},
        )
    ax.axhline(0, color="black", linestyle="--", linewidth=0.65)
    ax.set_xticks(x, NAMES)
    ax.margins(y=0.2)
    ax.set_ylabel("Economia frente ao completo (%)")
    first = ax.legend(
        handles=[Patch(facecolor=c, label=l) for c, l in zip(COLORS[1:], LABELS[1:])],
        loc="upper left",
        ncol=2,
        fontsize=7.3,
    )
    ax.add_artist(first)
    ax.legend(
        handles=[
            Patch(facecolor="white", edgecolor="#555", hatch=h, label=l)
            for h, l in (("//", "Raiz enviada"), ("", "Raiz omitida"))
        ],
        loc="lower left",
        ncol=2,
        fontsize=7,
        handlelength=1.4,
        columnspacing=0.7,
    )
    ax.set_title("Ambas as falhas em 100%", fontsize=8, pad=6)


def controls(diagnostic):
    from pqc_iot_simulator.pqc_iot_sim.crypto.tls.diagnostics import TicketDiagnostics

    return TicketDiagnostics(diagnostic).matched_client_controls()


def panel_c(ax, chain, comparisons, amort, amort_cmp, diagnostic):
    ec, ml = controls(diagnostic)
    x = np.arange(3)
    for j, (d, color, label) in enumerate(
        zip((ec, ml), COLORS[:2], ("Cliente ECDSA P-256", "Cliente ML-DSA-44"))
    ):
        c = d["connections"][0]
        ys = [
            d["certificates"]["client"]["der_bytes"],
            c["ticket_plaintext_session_der_bytes"],
            c["tickets_tx_bytes"][0],
        ]
        bars = ax.bar(x + (j - 0.5) * 0.36, ys, 0.34, color=color, label=label)
        for bar, y in zip(bars, ys):
            ax.text(
                bar.get_x() + bar.get_width() / 2,
                y + 70,
                f"{y:,}".replace(",", "."),
                ha="center",
                fontsize=7,
            )
    ax.set_xticks(x, ("Certificado\ndo cliente", "Estado\nda sessão", "Ticket"))
    ax.set_ylabel("Tamanho (bytes)")
    ax.margins(y=0.2)
    ax.set_ylim(bottom=0)
    ax.legend(fontsize=7.3, loc="upper left", ncol=1)
    ax.set_title("Servidor e CA com ML-DSA-44 mantidos", fontsize=8, pad=6)


def panel_d(ax, chain, comparisons, amort, amort_cmp, diagnostic):
    x = np.arange(3)
    for j, (policy, color, label) in enumerate(zip(POLICIES, COLORS, LABELS)):
        ys = [
            float(
                one(
                    amort, recovery=False, batch_size=b, profile="pqc", treatment=policy
                )["median_ipv4_bytes_per_confirmed_reading"]
            )
            / 1000
            for b in (1, 10, 100)
        ]
        ax.bar(x + (j - 1) * 0.25, ys, 0.23, color=color, label=label)
    ax.set_xticks(x, ("1", "10", "100"))
    ax.set_xlabel("Limite de leituras por conexão")
    ax.set_ylabel("IPv4 por leitura (kB)")
    ax.margins(y=0.2)
    ax.set_ylim(bottom=0)
    ax.legend(
        ncol=3, fontsize=7.2, loc="upper right", columnspacing=0.6, handlelength=1
    )
    key_title(ax)


def panel_e(ax, chain, comparisons, amort, amort_cmp, diagnostic):
    for policy, color, label in zip(POLICIES[1:], COLORS[1:], LABELS[1:]):
        rows = [
            one(amort_cmp, recovery=True, batch_size=b, profile="pqc", treatment=policy)
            for b in (1, 10, 100)
        ]
        ys = [float(r["median_savings_pct"]) for r in rows]
        errors = [
            [y - float(r["savings_ci_low"]) for y, r in zip(ys, rows)],
            [float(r["savings_ci_high"]) - y for y, r in zip(ys, rows)],
        ]
        ax.errorbar(
            (0, 1, 2),
            ys,
            yerr=errors,
            color=color,
            marker="o",
            markersize=4,
            label=label,
            linewidth=1.3,
            capsize=2,
        )
    ax.axhline(0, color="black", linestyle="--", linewidth=0.65)
    ax.set_xticks((0, 1, 2), ("1", "10", "100"))
    ax.set_xlabel("Limite de leituras por conexão")
    ax.set_ylabel("Economia frente ao completo (%)")
    ax.margins(y=0.2)
    ax.legend(ncol=2, fontsize=7.4, loc="upper right")
    key_title(ax)


def panel_f(ax, chain, comparisons, amort, amort_cmp, diagnostic):
    for recovery, offset, color, marker, label in (
        (False, -0.07, COLORS[0], "s", "Sem falhas"),
        (True, 0.07, COLORS[2], "o", "Recuperação"),
    ):
        rows = [
            one(
                amort_cmp,
                recovery=recovery,
                batch_size=b,
                profile="pqc",
                treatment="resume_preserved",
            )
            for b in (1, 10, 100)
        ]
        ys = [float(r["median_ms_difference_per_reading"]) for r in rows]
        errors = [
            [y - float(r["ms_ci_low"]) for y, r in zip(ys, rows)],
            [float(r["ms_ci_high"]) - y for y, r in zip(ys, rows)],
        ]
        ax.errorbar(
            np.arange(3) + offset,
            ys,
            yerr=errors,
            color=color,
            fmt=marker,
            markersize=4,
            capsize=3,
            label=label,
        )
    ax.axhline(0, color="black", linestyle="--", linewidth=0.65)
    ax.set_xticks((0, 1, 2), ("1", "10", "100"))
    ax.set_xlabel("Limite de leituras por conexão")
    ax.set_ylabel("Diferença de tempo (ms/leitura)")
    ax.margins(y=0.2)
    ax.legend(ncol=2, fontsize=7.4, loc="lower right")
    key_title(ax)


PANELS = (
    ("contexto_custos_sem_raiz", panel_a),
    ("contexto_economia_cadeias", panel_b),
    ("contexto_ticket_identidade", panel_c),
    ("contexto_leituras_custos", panel_d),
    ("contexto_leituras_recuperacao", panel_e),
    ("contexto_leituras_tempo", panel_f),
)


class ContextFigures:
    """Exporta seis painéis dos testes de contexto, sem executar novos experimentos.

    Configure once, then call ``run()`` from a notebook.
    """

    def __init__(
        self,
        *,
        data_dir=DATA / "context",
        diagnostic=DATA / "diagnostico_tickets_20261106.json",
        output=ROOT / "results/pesquisa/figuras",
    ):
        self.data_dir = Path(data_dir) if data_dir is not None else None
        self.diagnostic = Path(diagnostic) if diagnostic is not None else None
        self.output = Path(output) if output is not None else None

    def run(self):
        args = self
        args.output.mkdir(parents=True, exist_ok=True)
        paths = [args.data_dir / f for f in FILES] + [args.diagnostic]
        before = {p.name: sha(p) for p in paths}
        data = [read(p) for p in paths[:-1]] + [json.loads(args.diagnostic.read_text())]
        assert data[-1]["status"] == "PASS"
        assert len(data[0]) == 54 and len(data[2]) == 54
        blocks = {int(r["blocks"]) for r in data[0] + data[2]}
        if len(blocks) != 1:
            raise ValueError("chain and amortization block counts differ")
        blocks = blocks.pop()
        assert sum((int(r["executions"]) for r in data[0])) == 54 * blocks
        assert sum((int(r["executions"]) for r in data[2])) == 54 * blocks
        assert sum((int(r["confirmed_readings"]) for r in data[0])) == 486 * blocks
        assert sum((int(r["confirmed_readings"]) for r in data[2])) == 5400 * blocks
        assert all(
            (
                int(r["duplicate_effects"]) == 0 and int(r["unconfirmed_readings"]) == 0
                for r in data[0] + data[2]
            )
        )
        plt.rcParams.update(
            {
                "font.family": "DejaVu Sans",
                "font.size": 9,
                "axes.labelsize": 9,
                "xtick.labelsize": 7.3,
                "ytick.labelsize": 8.5,
                "pdf.fonttype": 42,
                "svg.fonttype": "none",
            }
        )
        for name, draw in PANELS:
            (fig, ax) = plt.subplots(figsize=(4, 2.6), layout="constrained")
            style(ax)
            draw(ax, *data)
            for ext in ("pdf", "png", "svg"):
                fig.savefig(args.output / (name + "." + ext), dpi=280)
            plt.close(fig)
        assert before == {p.name: sha(p) for p in paths}
        proof = {
            "status": "PASS",
            "inputs_sha256": before,
            "notebook_sha256": sha(ROOT / "Pesquisa_Retomada_Sessoes.ipynb"),
            "chain_executions": 54 * blocks,
            "chain_confirmed_readings": 486 * blocks,
            "amortization_executions": 54 * blocks,
            "amortization_confirmed_readings": 5400 * blocks,
            "blocks_per_condition": blocks,
            "scope": {
                "a": "Cadeias: 100%/100%, raiz omitida, mediana de bytes por leitura.",
                "b": "Cadeias: 100%/100%, medianas de comparações dentro da repetição; IC bootstrap exploratório 95%.",
                "c": "Controle local com exatamente mesmo servidor e CA; não é amostra de custo de rede.",
                "d": "100 leituras de 512 bytes, sem falhas, primeiro completo incluído, raiz omitida.",
                "e": "Mesma carga; reinício antes da 51ª leitura e supressão de sua primeira confirmação.",
                "f": "Contexto preservado menos completo, diferenças dentro do bloco e IC bootstrap exploratório 95%.",
            },
            "all_network_panels": {
                "packet_loss_pct": 0,
                "deduplication": True,
                "root_trust": "mantida",
            },
            "panels": [name for (name, _) in PANELS],
        }
        (args.output / "proveniencia.json").write_text(
            json.dumps(proof, ensure_ascii=False, indent=2) + "\n"
        )
        print(f"PASS: seis painéis; {blocks} blocos; dados de origem preservados.")
        return args.output


pasta_tabelas = estudo.directory / "context_analysis" if REPRODUZIR else reanalise.directory / "context_analysis"
diagnostico = estudo.directory / "identity_tickets/results.json" if REPRODUZIR else DATA / "diagnostico_tickets_20261106.json"
pasta_figuras = (estudo.directory if REPRODUZIR else reanalise.directory) / "figures"
ContextFigures(data_dir=pasta_tabelas, diagnostic=diagnostico, output=pasta_figuras).run()
for nome, _ in PANELS:
    display(Image(filename=str(pasta_figuras / f"{nome}.png"), width=720))


### Gráficos: falhas de estado e interrupções

As seleções de dados, escalas e comandos de desenho estão nesta célula. As figuras são calculadas a partir das tabelas e verificações, sem valores de resultado inseridos manualmente.

In [ ]:
"""Render article panels from frozen, verified recovery campaigns (no new trials)."""


import hashlib
import json
import statistics
from pathlib import Path

ROOT = Path.cwd()
POLICIES = [
    "checkpoint",
    "persist_each",
    "reserve_blocks",
    "full_pqc",
    "psk_resume",
    "pqc_resume",
]
LABELS = ["Checkpoint", "Por uso", "Blocos", "Completo", "PSK", "PSK+KEM"]
MODES = ["full_pqc", "psk_resume", "pqc_resume"]
NAMES = ["Completo", "PSK", "PSK+KEM"]
COLORS = ["#315a91", "#b56a29", "#227a65"]


def read_campaign(directory):
    directory = Path(directory)
    manifest = json.loads((directory / "manifest.json").read_text())
    for file, digest in manifest["source_sha256"].items():
        if hashlib.sha256((ROOT / file).read_bytes()).hexdigest() != digest:
            raise RuntimeError("source differs from frozen campaign: " + file)
    raw = (directory / "results.json").read_bytes()
    return json.loads(raw), hashlib.sha256(raw).hexdigest()


class PrototypeFigures:
    """Render article panels from frozen, verified recovery campaigns (no new trials).

    Configure once, then call ``run()`` from a notebook.
    """

    def __init__(self, *, security, transitions, output, tls=None):
        self.security = Path(security)
        self.transitions = Path(transitions)
        self.tls = Path(tls) if tls is not None else None
        self.output = Path(output)

    def run(self):
        args = self
        security, security_hash = read_campaign(args.security)
        transitions, transition_hash = read_campaign(args.transitions)
        tls, tls_hash = read_campaign(args.tls) if args.tls else (None, None)
        if (
            security["verdict"]["status"] != "SUPPORTED_IN_TESTED_MODEL"
            or transitions["assessment"]["status"] != "PASS"
        ):
            raise RuntimeError("failed experiments cannot support figures")
        if len(security["cases"]) != 240 or len(transitions["cases"]) != 330:
            raise ValueError(
                "prototype figures require the complete ten-repetition design"
            )
        if tls is not None and (tls["status"] != "PASS" or len(tls["cases"]) != 60):
            raise ValueError("legacy TLS figures require the complete 60-case pilot")
        import matplotlib

        matplotlib.use("Agg")
        import matplotlib.pyplot as plt

        plt.rcParams.update(
            {
                "font.size": 9,
                "axes.spines.top": False,
                "axes.spines.right": False,
                "svg.fonttype": "none",
                "savefig.dpi": 300,
            }
        )
        args.output.mkdir(parents=True, exist_ok=True)
        panels = []

        def canvas():
            (fig, ax) = plt.subplots(figsize=(4.6, 3.1), layout="constrained")
            ax.grid(axis="y", alpha=0.2)
            ax.set_axisbelow(True)
            return (fig, ax)

        def save(fig, name):
            for suffix in ("png", "svg"):
                fig.savefig(args.output / (name + "." + suffix))
            plt.close(fig)
            panels.append(name)

        def bars(name, labels, values, ylabel, colors=None, rotate=0):
            (fig, ax) = canvas()
            ax.bar(range(len(values)), values, color=colors or COLORS[0], width=0.65)
            ax.set_xticks(
                range(len(values)),
                labels,
                rotation=rotate,
                ha="right" if rotate else "center",
            )
            ax.set_ylabel(ylabel)
            ax.set_ylim(0, max(values, default=0) * 1.23 or 1)
            for i, value in enumerate(values):
                ax.annotate(
                    f"{value:g}",
                    (i, value),
                    xytext=(0, 4),
                    textcoords="offset points",
                    ha="center",
                )
            save(fig, name)

        security_summary = {}
        for fault in ("power_loss", "client_storage_rollback"):
            security_summary[fault] = []
            for policy in POLICIES:
                rows = [
                    r
                    for r in security["cases"]
                    if r["fault"] == fault and r["policy"] == policy
                ]
                assert len(rows) == 10
                security_summary[fault].append(
                    {
                        "policy": policy,
                        "reuses": sum((r["reused_key_nonce"] for r in rows)),
                        "plaintexts": sum(
                            (r["known_plaintext_recoveries"] for r in rows)
                        ),
                        "writes": statistics.median(
                            (r["steady_state_writes"] for r in rows)
                        ),
                    }
                )
        for name, fault, field, ylabel in [
            (
                "estado_energia",
                "power_loss",
                "reuses",
                "Reutilizações após queda de energia",
            ),
            (
                "estado_rollback",
                "client_storage_rollback",
                "reuses",
                "Reutilizações após rollback completo",
            ),
            (
                "estado_textos",
                "client_storage_rollback",
                "plaintexts",
                "Textos recuperados após rollback",
            ),
            (
                "estado_escritas",
                "power_loss",
                "writes",
                "Escritas na fase estável de 8 leituras",
            ),
        ]:
            bars(
                name,
                LABELS,
                [r[field] for r in security_summary[fault]],
                ylabel,
                rotate=30,
            )
        scenarios = list(dict.fromkeys((r["scenario"] for r in transitions["cases"])))
        scenario_labels = [
            "Sem falha",
            "ClientHello",
            "ServerHello",
            "ClientFinished",
            "Commit",
            "Confirmação",
            "Checkpoint",
            "Cifragem",
            "ACK perdido",
            "Truncado",
            "Raiz perdida",
        ]
        transition_summary = []
        for scenario in scenarios:
            row = {"scenario": scenario}
            for mode in MODES:
                selected = [
                    r
                    for r in transitions["cases"]
                    if r["scenario"] == scenario and r["policy"] == mode
                ]
                assert len(selected) == 10
                row[mode] = {
                    "bytes": statistics.median(
                        (r["total_handshake_bytes"] for r in selected)
                    ),
                    "fallbacks": sum((r["full_fallback"] for r in selected)),
                    "duplicates": sum((r["application_duplicates"] for r in selected)),
                }
            transition_summary.append(row)
        (fig, ax) = canvas()
        for mode, label, color in zip(MODES, NAMES, COLORS):
            ax.plot(
                range(len(scenarios)),
                [r[mode]["bytes"] / 1000 for r in transition_summary],
                "o-",
                label=label,
                color=color,
                markersize=3,
            )
        ax.set_xticks(range(len(scenarios)), list("ABCDEFGHIJK"))
        ax.set_xlabel("Ponto de interrupção (A-K)")
        ax.set_ylabel("Mediana de bytes enquadrados (mil)")
        ax.legend(fontsize=8)
        save(fig, "transicoes_bytes")
        selected_faults = transition_summary[-2:]
        bars(
            "transicoes_fallbacks",
            ["Checkpoint\ntruncado", "Raiz\nperdida"],
            [r["pqc_resume"]["fallbacks"] for r in selected_faults],
            "Fallbacks completos: PSK+KEM",
        )
        savings = [
            100 * (1 - r["pqc_resume"]["bytes"] / r["full_pqc"]["bytes"])
            for r in transition_summary
        ]
        (fig, ax) = canvas()
        indices = [0, 9, 10]
        values = [savings[i] for i in indices]
        ax.bar(range(3), values, color=[COLORS[2], COLORS[2], COLORS[1]])
        ax.axhline(0, color="#444444", linewidth=0.7)
        ax.set_xticks(range(3), ["Sem falha", "Truncado", "Raiz perdida"])
        ax.set_ylabel("Economia frente ao completo (%)")
        ax.set_ylim(-17, 82)
        for i, value in enumerate(values):
            ax.annotate(
                f"{value:.2f}%".replace(".", ","),
                (i, value),
                xytext=(0, 5 if value >= 0 else -13),
                textcoords="offset points",
                ha="center",
            )
        save(fig, "transicoes_economia")
        lost = next(
            (r for r in transition_summary if r["scenario"] == "lost_reading_ack")
        )
        bars(
            "transicoes_duplicados",
            NAMES,
            [lost[m]["duplicates"] for m in MODES],
            "Efeitos duplicados com ACK perdido",
            COLORS,
        )
        tls_summary, warm, duplicates = [], {}, []
        if tls is not None:
            cases = tls["cases"]
            tls_summary = []
            for loss in (0, 1, 5):
                for mode in ("full", "resume"):
                    selected = [
                        c
                        for c in cases
                        if c["loss_pct"] == loss
                        and c["mode"] == mode
                        and c["deduplicate"]
                    ]
                    assert len(selected) == 5
                    wire = [
                        c["capture"]["iot_tx_ip_bytes"]
                        + c["capture"]["iot_rx_ip_bytes"]
                        for c in selected
                    ]
                    row = {
                        "loss": loss,
                        "mode": mode,
                        "wire_median": statistics.median(wire),
                        "wire_min": min(wire),
                        "wire_max": max(wire),
                    }
                    for key in ("first_reading_ack_ms", "reading_recovery_ms"):
                        row[key] = statistics.median(
                            (
                                statistics.median(
                                    (r[key] for r in c["clients"] if r["acknowledged"])
                                )
                                for c in selected
                            )
                        )
                    row["tcp_retransmitted_segments"] = statistics.median(
                        (
                            sum(
                                (
                                    r["tcp_retransmitted_segments"]
                                    for r in c["clients"] + c["server_connections"]
                                )
                            )
                            for c in selected
                        )
                    )
                    tls_summary.append(row)
            warm = {
                mode: statistics.median(
                    (
                        r["handshake_tx_tls_bytes"] + r["handshake_rx_tls_bytes"]
                        for c in cases
                        if c["mode"] == mode
                        for r in c["clients"]
                        if r["boot"] == 1
                    )
                )
                for mode in ("full", "resume")
            }
            bars(
                "tls_handshake",
                ["Completo", "Retomada"],
                list(warm.values()),
                "Mediana de bytes TLS até SSL_connect",
                [COLORS[0], COLORS[2]],
            )
            for name, field, ylabel in [
                ("tls_ipv4", "wire_median", "Bytes IPv4 por ensaio (mil)"),
                (
                    "tls_ack",
                    "first_reading_ack_ms",
                    "Confirmação após TCP conectado (ms)",
                ),
                (
                    "tls_recuperacao",
                    "reading_recovery_ms",
                    "Recuperação da leitura no host (ms)",
                ),
                (
                    "tls_retransmissoes",
                    "tcp_retransmitted_segments",
                    "Segmentos TCP retransmitidos por ensaio",
                ),
            ]:
                (fig, ax) = canvas()
                for i, (mode, label, color) in enumerate(
                    zip(
                        ("full", "resume"),
                        ("Completo", "Retomada"),
                        (COLORS[0], COLORS[2]),
                    )
                ):
                    rows = [r for r in tls_summary if r["mode"] == mode]
                    scale = 1000 if field == "wire_median" else 1
                    x = [j + (i - 0.5) * 0.35 for j in range(3)]
                    values = [r[field] / scale for r in rows]
                    error = (
                        [
                            [v - r["wire_min"] / scale for (v, r) in zip(values, rows)],
                            [r["wire_max"] / scale - v for (v, r) in zip(values, rows)],
                        ]
                        if field == "wire_median"
                        else None
                    )
                    ax.bar(
                        x,
                        values,
                        width=0.35,
                        label=label,
                        color=color,
                        yerr=error,
                        capsize=3,
                    )
                ax.set_xticks(range(3), ["0%", "1%", "5%"])
                ax.set_xlabel("Perda configurada por direção")
                ax.set_ylabel(ylabel)
                ax.legend(fontsize=8)
                save(fig, name)
            duplicates = [
                sum(
                    (
                        c["application"]["duplicate_effects"]
                        for c in cases
                        if c["deduplicate"] == d
                    )
                )
                for d in (False, True)
            ]
            bars(
                "tls_duplicados",
                ["Desativada", "Durável"],
                duplicates,
                "Efeitos duplicados no conjunto de ensaios",
                [COLORS[1], COLORS[2]],
            )
        facts = {
            "source_sha256": {
                "security": security_hash,
                "transitions": transition_hash,
                "tls": tls_hash,
            },
            "security": security_summary,
            "transitions": transition_summary,
            "scenario_labels": scenario_labels,
            "tls": tls_summary,
            "tls_warm_bytes": warm,
            "tls_duplicates": duplicates,
            "panels": panels,
            "statistics": "Ten prototype repetitions; optional legacy TLS pilot uses five blocks; descriptive medians.",
        }
        (args.output / "dados_figuras.json").write_text(
            json.dumps(facts, indent=2, ensure_ascii=False)
        )
        print(
            json.dumps(
                {
                    "panels": len(panels),
                    "warm_tls_bytes": warm,
                    "duplicate_effects": duplicates,
                    "sources": facts["source_sha256"],
                },
                indent=2,
            )
        )
        return args.output


if REPRODUZIR:
    pasta_prototipo = estudo.directory / "prototype_figures"
    PrototypeFigures(security=estudo.directory / "prototype", transitions=estudo.directory / "transitions", output=pasta_prototipo).run()
    for arquivo in sorted(pasta_prototipo.glob("*.png")):
        display(Image(filename=str(arquivo), width=720))


## 8. Organização e reprodução

- **Simulação:** `Simulation`, com `session_recovery()` para configurar os cenários.
- **Execução:** classes em `control/` e `runtime/`; métricas em `utils/`.
- **Gráficos:** código nas células da seção 7 deste notebook.
- **Dados históricos para reanálise:** `pqc_iot_simulator/pqc_iot_sim/data/`.
- **Dados novos:** `estudo.directory`, incluindo resultados, configurações, capturas e comparações.

A comparação com o artigo é salva em `estudo.directory / "article_comparison"`. Os textos do artigo e os relatórios anteriores à limpeza permanecem no histórico Git, commit `76bf3cd`. As novas execuções não sobrescrevem as observações históricas.
